# info
This notebook contains tests and working notes for the energy efficiency indicator (EEI) for for CO2 injection in petroleum fields on the NCS.  
The code idea: 
1. get a rough estimate of the wellhead pressure (WHP) required to inject CO2 at a given depth,  BHP, and temperature as:  
$\text{WHP} = \text{BHP} + [\text{friction losses}] - [\text{column weight}]$
2. calculate specific isothermal compression work for WHP

### main conclusions as of 2026.08.26
1. Required CO₂ injection pressure at the wellhead (WHP) depends on:
   - The current reservoir pressure which is currently unknown (very strongly)
   - Wellbore temperature profile which is hard to establish without detailed modelling (strongly)
   - Friction losses (moderately)

2. Minimal CO₂ compression work depends approximately linearly on WHP, with a much smaller range of variation (200 – 240 MJ/tonne vs. 50 – 450 bar).

3. Specific compression work **w(WHP)** is too uncertain to be used as an energy efficiency indicator.

4. The following, levelized, form of energy efficiency indicator is preferable, but more data/work needed:

$$w^*(t) = \frac{1}{Q(t)} \int_0^t w(p_{wh}(t)) \, dQ$$

**Legend:**
- **$w$** – specific compression work
- **$w^*$** – levelized specific compression work
- **$p_{wh}$** – wellhead pressure
- **$Q(t)$** – cumulative CO2 injection

# packages, data

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, TheilSenRegressor
import os,sys
from datetime import datetime, date
# import npd_wraper as npd
import plotly.express as px
import plotly.graph_objects as go  
import colorsys, json
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d, RegularGridInterpolator

# The unicellar is available at https://github.com/alex11818/unicellar
from submodules.unicellar.unicellar.plotters import multiplot
from submodules.unicellar.unicellar.helpers \
    import get_fluid_properties, estimate_thp, estimate_bhp, estimate_thp, \
        estimate_compression_work
# from submodules.unicellar import unicellar as uc
from utils import generate_rainbow_colors

In [2]:
df = pd.read_csv('./data/v2026.csv', index_col=0)

# compression work

In [3]:
# CO2 specific isothermal compression work from 1 bar (MJ/tonne)
pp = list(range(1, 200+1, 1))
ww, qq = estimate_compression_work(fluid='CO2', p=pp, degC=20, p_inc=1)
fig=px.line(x=pp, y=ww/1000, template='plotly_white')
# fig.update_yaxes(title='J/kg')
fig.update_yaxes(title='MJ/tonne')
fig.update_xaxes(title='target pressure (bar)', range=[0,200])
fig.update_layout(font_size=16, margin_pad=0)
fig.show(renderer='browser')
# fig.show()

In [4]:
estimate_compression_work(fluid='CO2', p=100, degC=20)

(np.float64(210105.89049999992), np.float64(469001.8904999999))

In [5]:
estimate_compression_work(fluid='CO2', p0=100, p=200, degC=20)

(np.float64(11104.199999999997), np.float64(19934.199999999997))

# static WHP estimates

## two ways to handle duplicated pressures

In [6]:
degC = 20
xstr = 'density (kg/m3)'
# xstr = 'pressure (bar)'
ystr = 'depth (m)'
# fields = ['TROLL', 'GULLFAKS', 'VOLVE','ULA','KVITEBJØRN']
fields = ['TROLL']
field_colors = generate_rainbow_colors(len(fields))

fig = go.Figure()
for f,c in zip(fields, field_colors):
    p0,z,p_h = df.loc[f,['p0','depth','p_h']]
    bhp = min(p0*1.0, 1011)
    for s,dsh in zip(['adjust',False], ['solid','longdash']):

        zdp=estimate_thp(
            fluid='CO2', bhp=bhp, tvd=z, degC=degC, all_points=True,
            pressure_duplicates=s)

        zdp[xstr] = zdp[xstr].astype(float)
        zdp[ystr] = zdp[ystr].astype(float)

        # if s is False: continue 
        x_max = zdp[xstr].max().round(0)

        fig.add_trace(go.Scatter(
            x=zdp[xstr], y=zdp[ystr], mode='lines', 
            line=dict(width=3, dash=dsh), 
            legendgroup=f, legendgrouptitle_text=f,
            name=f'pressure_duplicates={s}'
        ))

    fig.add_annotation(
        x=x_max, y=z+50, text=f, showarrow=False, arrowhead=1
    )

fig.update_yaxes(autorange='reversed', title=ystr)
fig.update_xaxes(title=xstr, range=[0, 1000]
                 )

fig.update_layout(
    # title=f'wellbore pressure profiles (BHP=1.2*p0, q=0, T=const={degC} C) vs. depth for selected fields', 
    # template='plotly_dark',
    template='plotly_white', font_size = 14,
    legend=dict(yanchor='bottom', y=0.02, xanchor='left', x=0.02),
)
fig.show(renderer='browser')

## wellbore pressure profiles

In [7]:
degC = 20
tvd = 5000
bhp = 0.101*tvd
# xstr = 'density (kg/m3)'
xstr = 'pressure (bar)'
ystr = 'depth (m)'
fields = ['TROLL', 'GULLFAKS', 'VOLVE','ULA','KVITEBJØRN']
field_colors = generate_rainbow_colors(len(fields))
fig = go.Figure()

for f,c in zip(fields, field_colors):
    p0,z,p_h = df.loc[f,['p0','depth','p_h']]
    bhp = min(p0*1.2, 1011)
    for s,dsh in zip(['adjust',False], ['solid','longdash']):

        zdp=estimate_thp(
            fluid='CO2', bhp=bhp, tvd=z, degC=degC, all_points=True,
            pressure_duplicates=s)

        zdp[xstr] = zdp[xstr].astype(float)
        zdp[ystr] = zdp[ystr].astype(float)

        if s is False: continue 
        x_max = zdp[xstr].max().round(0)

        fig.add_trace(go.Scatter(
            x=zdp[xstr], y=zdp[ystr], mode='lines', 
            line=dict(width=3, dash=dsh, color=c), 
            legendgroup=f,
            name='wellbore'
        ))

    # initial reservoir pressure point
    fig.add_trace(go.Scatter(
        x=[p0], y=[z], mode='markers',
        marker=dict(size=10, color=c, line=dict(width=1, color='black')),
        legendgroup=f, legendgrouptitle_text=f,
        name='p0', showlegend=True
    ))

    fig.add_trace(go.Scatter(
        x=[p0, x_max], y=[z, z], mode='lines',
        line=dict(width=1, color=c, dash='dash'), showlegend=False,
        legendgroup=f, legendgrouptitle_text=f,
    ))


    fig.add_annotation(
        x=(bhp+p0)/2, y=z+50, text=f, showarrow=False, arrowhead=1
    )

fig.update_yaxes(autorange='reversed', title=ystr)
fig.update_xaxes(title=xstr, range=[0, 1000])

fig.update_layout(
    title=f'wellbore pressure profiles (BHP=1.2*p0, q=0, T=const={degC} C) vs. depth for selected fields', 
    # template='plotly_dark',
    template='plotly_white',    
    legend=dict(yanchor='bottom', y=0.02, xanchor='left', x=0.02),
    font_size = 14
)
fig.show(renderer='browser')

## static pressure vs. depth

In [8]:
# compact_legend=False: per-field legends (legend + legend2); True: single trace-type legend (legend3)
compact_legend = True

xstr = 'pressure (bar)'
ystr = 'depth (m)'
fields = ['TROLL', 'GULLFAKS', 'VOLVE','ULA','KVITEBJØRN']
field_colors = generate_rainbow_colors(len(fields))
style_specs = [(20, 1.2, 'solid')]
style_specs = [(20, 1.2, 'solid'), (50, 1.2, 'dash')]
style_specs = [(20, 1.2, 'solid'), (20, 1.0, 'dot')]
# style_specs = [(20, 1.2, 'solid'), (50, 1.2, 'dash'), (20, 1.0, 'dot')]
trace_count = len(style_specs)*len(style_specs[0])
field_count = len(fields)

fig = go.Figure()
for i, (f,c) in enumerate(zip(fields, field_colors)):
    legend_id = 'legend3' if compact_legend else ('legend' if i < 3 else 'legend2')
    p0,z,p_h = df.loc[f,['p0','depth','p_h']]
    for degC, p_mult, dsh in style_specs:
        bhp = min(p0*p_mult, 1011)
        zdp=estimate_thp(
            fluid='CO2', bhp=bhp, tvd=z, degC=degC, all_points=True)

        zdp[xstr] = zdp[xstr].astype(float)
        zdp[ystr] = zdp[ystr].astype(float)

        name = f'T={degC}°C, BHP={p_mult}p0'\
              if p_mult > 1.0 else f'T={degC}°C, BHP=p0'

        fig.add_trace(go.Scatter(
            x=zdp[xstr], y=zdp[ystr], mode='lines', 
            line=dict(width=3, dash=dsh, color=c), 
            legend=legend_id, legendgroup=f, legendgrouptitle_text=f,
            name=name, showlegend=(not compact_legend)
        ))
    # initial reservoir pressure point
    fig.add_trace(go.Scatter(
        x=[p0], y=[z], mode='markers', marker=dict(size=12, color=c),
        legend=legend_id, legendgroup=f, name='p0', showlegend=(not compact_legend)
    ))
    fig.add_annotation(
        x=1.1*p0, y=z+90, text=f, showarrow=False, arrowhead=1, font=dict(color=c))

if compact_legend:
    # dummy traces showing what each trace type looks like (p0 marker + line styles)
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers', marker=dict(size=12, color='white'),
        legend='legend3', name='initial pressure p0', 
        showlegend=True
    ))
    for degC, p_mult, dsh in style_specs:
        name = f'T={degC}°C, BHP={p_mult}p0' if p_mult > 1.0 else f'T={degC}°C, BHP=p0'
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines',
            line=dict(width=3, dash=dsh, color='white'),
            legend='legend3', name=name, showlegend=True
        ))

fig.update_yaxes(autorange='reversed', title=ystr)
fig.update_xaxes(title=xstr, range=[0, 1000])

legend_layout = (
    dict(legend3=dict(yanchor='top',y=0.95,xanchor='right',x=0.99,
                      font_size=28))
    if compact_legend else
    dict(legend =dict(yanchor='bottom',y=0.02,xanchor='left',x=0.02, 
                      font_size=20),
         legend2=dict(yanchor='top',y=0.99,xanchor='right',x=0.99,
                      font_size=20))
)
fig.update_layout(
    template='plotly_dark',
    **legend_layout,
    font_size = 20,  
)
# fig.show(renderer='browser')
fig.write_html('temp.html',auto_open=True)

## st. pressure vs. depth vs. reservoir pressure

In [9]:
# compact_legend=False: per-field legends (legend + legend2); True: single trace-type legend (legend3)
compact_legend = False

xstr = 'pressure (bar)'
ystr = 'depth (m)'
fields = ['TROLL', 'GULLFAKS', 'VOLVE','ULA','KVITEBJØRN']
field_colors = generate_rainbow_colors(len(fields))
style_specs = [(20, 1.20, 'solid'), (20, 1.00, 'dash'), 
               (20, 0.75, 'dot'),   (20, 0.50, 'dashdot')]
trace_count = len(style_specs)*len(style_specs[0])
field_count = len(fields)
fields = ['KVITEBJØRN']
fig = go.Figure()
for i, (f,c) in enumerate(zip(fields, field_colors)):
    legend_id = 'legend3' if compact_legend else ('legend' if i < 3 else 'legend2')
    p0,z,p_h = df.loc[f,['p0','depth','p_h']]
    for degC, p_mult, dsh in style_specs:
        bhp = p0*p_mult
        zdp=estimate_thp(
            fluid='CO2', bhp=bhp, tvd=z, degC=degC, all_points=True)

        zdp[xstr] = zdp[xstr].astype(float)
        zdp[ystr] = zdp[ystr].astype(float)

        name = f'T={degC}°C, BHP={p_mult:.2f}*p0'

        fig.add_trace(go.Scatter(
            x=zdp[xstr], y=zdp[ystr], mode='lines', 
            line=dict(width=3, dash=dsh, color=c), 
            legend=legend_id, legendgroup=f, legendgrouptitle_text=f,
            name=name, showlegend=(not compact_legend)
        ))
        # reservoir pressure point
        fig.add_trace(go.Scatter(
            x=[bhp], y=[z], mode='markers', marker=dict(size=12, color=c),
            legend=legend_id, legendgroup=f, name='bhp', showlegend=False
        ))
        fig.add_annotation(x=bhp, y=z+50, 
                           text=f'{p_mult:.0%} p0', 
                        #    text=f + f'<br>{p_mult:.0%} of p0', 
                           showarrow=False, yanchor='top',
                           font=dict(color=c))        

if compact_legend:
    # dummy traces showing what each trace type looks like (p0 marker + line styles)
    for degC, p_mult, dsh in style_specs:
        name = f'T={degC}°C, BHP={p_mult:.2f}*p0'
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines',
            line=dict(width=3, dash=dsh, color='white'),
            legend='legend3', name=name, showlegend=True
        ))
fig.update_yaxes(autorange='reversed', title=ystr)
fig.update_xaxes(title=xstr, range=[0, 1000])

legend_layout = (
    dict(legend3=dict(yanchor='top',y=0.95,xanchor='right',x=0.99,
                      font_size=28))
    if compact_legend else
    dict(legend =dict(yanchor='bottom',y=0.02,xanchor='left',x=0.02, 
                      font_size=20),
         legend2=dict(yanchor='top',y=0.99,xanchor='right',x=0.99,
                      font_size=20))
)
fig.update_layout(
    template='plotly_dark',
    **legend_layout,
    font_size = 20,  
)
# fig.show(renderer='browser')
fig.write_html('temp.html',auto_open=True)

## CO2 density vs. depth

In [10]:
# compact_legend=False: per-field legends (legend + legend2); True: single trace-type legend (legend3)
compact_legend = True
xstr = 'density (kg/m3)'
ystr = 'depth (m)'
fields = ['TROLL', 'GULLFAKS', 'VOLVE','ULA','KVITEBJØRN']
field_colors = generate_rainbow_colors(len(fields))
# style_specs = [(20, 1.2, 'solid')]
style_specs = [(20, 1.2, 'solid'), (50, 1.2, 'dash'), (20, 1.0, 'dot')]
# style_specs = [(20, 1.2, 'solid'), (50, 1.2, 'dash')]
# style_specs = [(20, 1.2, 'solid'), (20, 1.0, 'dot')]
# style_specs = [(20, 1.0, 'dot')]
# style_specs = [(50, 1.2, 'dash')]
trace_count = len(style_specs)*len(style_specs[0])
field_count = len(fields)

fig = go.Figure()
for i, (f,c) in enumerate(zip(fields, field_colors)):
    legend_id = 'legend3' if compact_legend else ('legend' if i < 3 else 'legend2')
    p0,z,p_h = df.loc[f,['p0','depth','p_h']]

    if f not in ['TROLL','GULLFAKS','KVITEBJØRN']: continue  # only KVITEBJØRN has a CO2 column, so skip other fields
    for degC, p_mult, dsh in style_specs:
        # bhp = min(p0*p_mult, 1011)
        bhp = p0*p_mult
        zdp=estimate_thp(
            fluid='CO2', bhp=bhp, tvd=z, degC=degC, all_points=True)

        zdp[xstr] = zdp[xstr].astype(float)
        zdp[ystr] = zdp[ystr].astype(float)

        name = f'T={degC}°C, BHP={p_mult}p0'\
              if p_mult > 1.0 else f'T={degC}°C, BHP=p0'

        fig.add_trace(go.Scatter(
            x=zdp[xstr], y=zdp[ystr], mode='lines', 
            line=dict(width=3, dash=dsh, color=c), 
            legend=legend_id, legendgroup=f, legendgrouptitle_text=f,
            name=name, showlegend=(not compact_legend)
        ))
    # initial reservoir pressure point (density at p0, since the x-axis is density)
    rho0 = zdp[xstr].iloc[0]  # density at p0
    # fig.add_trace(go.Scatter(
    #     x=[rho0], y=[z], mode='markers', marker=dict(size=12, color=c),
    #     legend=legend_id, legendgroup=f, name='p0', showlegend=(not compact_legend)
    # ))
    fig.add_annotation(
        x=rho0, y=z+90, text=f, showarrow=False, arrowhead=1, font=dict(color=c))

if compact_legend:
    # dummy traces showing what each trace type looks like (p0 marker + line styles)
    # fig.add_trace(go.Scatter(
    #     x=[None], y=[None], mode='markers', marker=dict(size=12, color='white'),
    #     legend='legend3', name='initial pressure p0', 
    #     showlegend=True
    # ))
    for degC, p_mult, dsh in style_specs:
        name = f'T={degC}°C, BHP={p_mult}p0' if p_mult > 1.0 else f'T={degC}°C, BHP=p0'
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines',
            line=dict(width=3, dash=dsh, color='white'),
            legend='legend3', name=name, showlegend=True
        ))

fig.update_yaxes(autorange='reversed', title=ystr)
fig.update_xaxes(title=xstr, range=[0, 1200])

legend_layout = (
    dict(legend3=dict(yanchor='bottom',y=0.02,xanchor='left',x=0.02, 
                      font_size=20))
    if compact_legend else
    dict(legend =dict(yanchor='bottom',y=0.02,xanchor='left',x=0.02, 
                      font_size=20),
         legend2=dict(yanchor='top',y=0.99,xanchor='right',x=0.99,
                      font_size=20))
)
fig.update_layout(
    template='plotly_dark',
    **legend_layout,
    # showlegend=False,
    font_size = 20,  
)
fig.write_html('temp_density.html', auto_open=True)

In [11]:
zz = df['depth'].sort_values()
for name, i in zz.items():
    print(f'{name:20s} {i:10.3f}')

TROLL                  1557.500
DRAUGEN                1630.000
GOLIAT                 1717.000
BREIDABLIKK            1741.800
BALDER                 1745.000
SVALIN                 1755.500
GRANE                  1756.600
SYMRA                  1800.000
JOHAN CASTBERG         1825.400
FRIGG                  1875.000
EDVARD GRIEG           1884.400
JOHAN SVERDRUP         1905.700
SOLVEIG                1933.000
HUGIN                  1936.300
RINGHORNE ØST          1936.700
VOLUND                 1945.000
URD                    1953.000
NORDØST FRIGG          1955.500
BØYLA                  1957.600
GULLFAKS               1994.000
JETTE                  2068.000
ØST FRIGG              2076.000
JOTUN                  2078.600
ODIN                   2082.100
ENOCH                  2083.300
HEIMDAL                2092.300
SKOGUL                 2109.500
ALVHEIM                2125.000
HYME                   2131.500
BESTLA                 2150.000
VILJE                  2150.000
GLITNE  

# adding friction

In [12]:
def friction(mq=31.7, D=0.157, rho=640, mu=0.06, L=1000, roughness=3e-5, 
             output_inputs=False):
    """
    Frictional pressure loss for single-phase flow in a circular pipe.
 
    Darcy-Weisbach with the Haaland (1983) explicit friction factor. Written for
    fixed mass rate, so the loss scales as D**-5:
 
        dp = 8 * f * mq**2 * L / (rho * pi**2 * D**5)
 
    Valid for turbulent, single-phase flow (Re > ~4000). Properties are taken as
    constant over the length L, so for a long well either use mid-depth values or
    call this per segment and sum.
 
    All arguments accept NumPy arrays and are broadcast against one another, so
    multi-dimensional sensitivity grids can be evaluated in a single call. See
    Examples.
 
    Parameters
    ----------
    mq : float or array_like
        Mass flow rate [kg/s]. For an annual tonnage, mq = Mt_per_yr * 1e9 /
        (365.25 * 24 * 3600). 1 Mt/yr = 31.7 kg/s.
    D : float or array_like
        Pipe INNER diameter [m]. Not the nominal or outer size - the D**-5
        scaling makes this the single most sensitive input. 7"
        tubing  (29 lb/ft) has an ID of 0.15707 m (6.184").
    rho : float or array_like
        Fluid density [kg/m3] at the local pressure and temperature. From your
        PVT table. Dense-phase CO2 in a well spans roughly 600-900.
    mu : float or array_like
        Dynamic viscosity [cP]. Dense-phase CO2 is of order 0.05-0.09 cP.
        Note 1 cP = 1e-3 Pa.s. Enters only through Re, so a 30% error moves
        the result by under 2% in the fully rough regime.
    L : float or array_like
        Pipe length along the flow path [m], i.e. measured depth, not TVD. Pass
        L = 1000.0 to read the answer directly as a gradient per kilometre.
    roughness : float or array_like, optional
        Absolute wall roughness [m]. Default 3e-5 (30 micron), typical of new
        carbon-steel tubing. Plausible range 1.5e-5 to 5e-5, which swings f by
        about +/-12% - in practice the dominant uncertainty in this calculation,
        larger than the choice of friction correlation.
 
    Returns
    -------
    dict
        v         : bulk velocity [m/s]
        Re        : Reynolds number [-]
        f         : Darcy friction factor [-]. Fanning = f / 4.
        dp_bar    : frictional pressure loss over L [bar]
        kJ_per_kg : specific energy dissipated, dp / rho [kJ/kg]
        kWh_per_t : the same, per tonne of fluid [kWh/tonne]
 
        Each value has the broadcast shape of the inputs; scalar inputs give
        NumPy scalars.
 
    Raises
    ------
    ValueError
        If any element of Re falls outside 1e4 to 1e8. This is a unit sentinel,
        not a physical limit: laminar flow needs about 4e-4 Mt/yr in 5-1/2"
        tubing, so out-of-range Re in practice means mq, D or mu was passed
        in the wrong units. Note that Haaland returns a plausible-looking number
        at low Re rather than failing, and that f is nearly independent of Re in
        the fully rough regime, so a 1000x viscosity error shifts dp by only ~1%
        and would otherwise pass unnoticed.
 
    Notes
    -----
    This is the friction term only. The net wellhead pressure requirement is
    hydrostatic gain minus this loss minus the reservoir drawdown see
    estimate_thp_friction.
 
    Haaland agrees with the implicit Colebrook-White equation to within ~1.5%
    worst case, and to ~0.03% at Re ~ 5e6, so the explicit form costs nothing
    here while remaining vectorisable over NumPy arrays.
 
    Because v = 4*mq/(rho*pi*D**2), density cancels out of the Reynolds
    number entirely: Re = 4*mq/(pi*D*mu). Re therefore depends only on mass
    rate, diameter and viscosity.
 
    Examples
    --------
    Scalar call:
 
    >>> mq = 1e9 / (365.25 * 24 * 3600)          # 1 Mt/yr
    >>> r = friction(mq, D=0.12426, rho=700, mu=0.06, L=2600)
    >>> float(round(r['dp_bar'], 2)), float(round(r['kWh_per_t'], 3))
    (14.74, 0.585)
 
    Vector over one axis:
 
    >>> r = friction(mq, D=0.12426, rho=np.array([620, 700, 780]),
    ...              mu=0.06, L=1000.0)
    >>> np.round(r['dp_bar'], 2)
    array([6.4 , 5.67, 5.09])
 
    Two-dimensional grid by broadcasting a column against a row:
 
    >>> rho = np.array([620, 700, 780, 860])[:, None]      # rows
    >>> D   = np.array([0.11862, 0.12426, 0.15706])        # columns
    >>> np.round(friction(mq, D, rho, 0.06, 1000.0)['dp_bar'], 2)
    array([[8.15, 6.4 , 1.9 ],
           [7.22, 5.67, 1.69],
           [6.48, 5.09, 1.51],
           [5.87, 4.62, 1.37]])
    """
    mu = np.asarray(mu, dtype=float) * 1e-3          # cP -> Pa.s
    mq, D, rho, L = (np.asarray(x, dtype=float) for x in (mq, D, rho, L))
    roughness = np.asarray(roughness, dtype=float)
 
    v = 4 * mq / (rho * np.pi * D**2)
    Re = rho * v * D / mu
 
    bad = (Re <= 1e4) | (Re >= 1e8)
    if np.any(bad):
        offending = Re[bad] if Re.ndim else Re
        n_bad, n_tot = int(np.sum(bad)), int(np.size(Re))
        raise ValueError(
            f"Re out of range for {n_bad} of {n_tot} case(s): "
            f"min {np.min(offending):.3e}, max {np.max(offending):.3e}. "
            f"Expected 1e4 to 1e8; injection wells at commercial rates sit at "
            f"Re ~1e6-1e7. Check units: mu in centipoise (CO2 ~0.06), "
            f"D the INNER diameter in metres, mq in kg/s."
        )
 
    f = (-1.8 * np.log10((roughness / D / 3.7)**1.11 + 6.9 / Re))**-2
    dp = 8 * f * mq**2 * L / (rho * np.pi**2 * D**5)

    output=dict(
        v=v, Re=Re, f=f, dp_bar=dp / 1e5,
        kJ_per_kg=dp / rho / 1e3, kWh_per_t=dp / rho / 3.6e3)
    if output_inputs:
        inputs=dict(mq=mq, D=D, rho=rho, mu=mu, L=L, roughness=roughness)
        output.update(inputs)
        # if isinstance(output_inputs, bool):
        #     inputs=dict(mq=mq, D=D, rho=rho, mu=mu, L=L, roughness=roughness)
        #     output.update(inputs)
        # elif isinstance(output_inputs, str):
        #     eval(f'output[''{output_inputs}'']={output_inputs}') 
        # elif isinstance(output_inputs, list):
        #     for i in output_inputs:
        #         eval(f'output["{i}"]={i}')
        # else:
        #     pass

    return output

mq = 1e9 / (365.25 * 24 * 3600)                          # 1 Mt/yr -> kg/s
friction(mq, D=0.12426, rho=700, mu=0.06, L=2600)

{'v': np.float64(3.7328918957939115),
 'Re': np.float64(5411573.381332433),
 'f': np.float64(0.014447730859015516),
 'dp_bar': np.float64(14.743497203618155),
 'kJ_per_kg': np.float64(2.106213886231165),
 'kWh_per_t': np.float64(0.5850594128419903)}

In [13]:
MU = np.array([0.06, 0.08])
foo = friction(mq, D=0.12426, rho=[700, 900], mu=MU, L=1000, output_inputs=True)
pd.DataFrame(data=foo)

,v,Re,f,dp_bar,kJ_per_kg,kWh_per_t,mq,D,rho,mu,L,roughness
0,3.732892,5.411573e+06,0.014448,5.670576,0.810082,0.225023,31.688088,0.12426,700.0,0.00006,1000.0,0.00003
1,2.903360,4.058680e+06,0.014496,4.425107,0.491679,0.136577,31.688088,0.12426,900.0,0.00008,1000.0,0.00003


In [14]:
foo = friction(roughness=[3e-5, 5e-5, 10e-5], output_inputs='roughness')
pd.DataFrame(data=foo)

,v,Re,f,dp_bar,kJ_per_kg,kWh_per_t,mq,D,rho,mu,L,roughness
0,2.558528,4.284681e+06,0.013853,1.848281,0.288794,0.080221,31.7,0.157,640.0,0.00006,1000.0,0.00003
1,2.558528,4.284681e+06,0.015309,2.042535,0.319146,0.088652,31.7,0.157,640.0,0.00006,1000.0,0.00005
2,2.558528,4.284681e+06,0.017769,2.370744,0.370429,0.102897,31.7,0.157,640.0,0.00006,1000.0,0.00010


In [15]:
mq  = 1e9/(365.25*24*3600)          # kg/s, ~1 Mt/yr
mu    = 6e-2                          # cP
sizes = {'4.670" (23 lb/ft)':      0.11862,
         '4.892" (17 lb/ft)':      0.12426,
         '6.184" (7", 29 lb/ft)':  0.15706}

print("Frictional pressure gradient, dp/ds [bar/km]")
print(f"CO2 at {mq:.1f} kg/s (1.0 Mt/yr), mu = {mu*1e6:.0f} uPa.s\n")

hdr = f'{"rho [kg/m3]":>12s} | ' + " | ".join(f"{k:>22s}" for k in sizes)
print(hdr)
print("-" * len(hdr))

for rho in [620, 700, 780, 860]:
    vals = [friction(mq, D, rho, mu, 1000.0)["dp_bar"] for D in sizes.values()]
    print(f"{rho:>12d} | " + " | ".join(f"{x:>22.2f}" for x in vals))

Frictional pressure gradient, dp/ds [bar/km]
CO2 at 31.7 kg/s (1.0 Mt/yr), mu = 60000 uPa.s

 rho [kg/m3] |      4.670" (23 lb/ft) |      4.892" (17 lb/ft) |  6.184" (7", 29 lb/ft)
---------------------------------------------------------------------------------------
         620 |                   8.15 |                   6.40 |                   1.90
         700 |                   7.22 |                   5.67 |                   1.69
         780 |                   6.48 |                   5.09 |                   1.51
         860 |                   5.87 |                   4.62 |                   1.37


In [16]:
def estimate_thp_friction(
        fluid='CO2', bhp=100, tvd=1000, degC=20, mq=31.7, \
        D=0.157, roughness=5e-5, p_max=1011):
    """
    Rough estimate of Tubing Head Pressure (THP) for a given Bottom Hole
    Pressure (BHP) in a vertical injection well, under static and flowing
    (with friction) conditions:

    (1) Static pressure profile — fluid column weight only, no friction.
    (2) Flowing pressure profile — at mass rate mq, with frictional loss
        accumulated from bottomhole upward:

        p(z) = BHP - column_weight(z to tvd) + friction(z to tvd)

    This function first calls estimate_thp() to get the static pressure
    profile, then evaluates friction() on each segment and accumulates the
    losses from tvd upward to the wellhead.

    Assumptions:

    - Isothermal at degC.
    - Single-phase, fully turbulent flow (Re > 1e4).
    - Density varies with pressure (from NIST tables) but is evaluated on
      the frictionless pressure profile, not updated for the friction
      correction.

    Parameters
    ----------
    fluid : str
        Fluid name/formula, passed through to estimate_thp() (default 'CO2').
    bhp : float, int
        Bottom Hole Pressure (bar).
    tvd : float, int
        Well True Vertical Depth (m).
    degC : float, int
        Representative (constant) temperature (degC).
    mq : float, int
        Mass flow rate (kg/s), passed to friction(); 1 Mt/yr = 31.7 kg/s.
    D : float, int
        Pipe inner diameter (m), passed to friction(). Friction scales as
        D**-5, so this is the most sensitive input.
    roughness : float, int
        Absolute pipe wall roughness (m), passed to friction(). The plausible
        range is 1.5e-5 to 5e-5.
    p_max : float, int
        maximum pressure (bar) passed to get_fluid_properties(); raise
        this for ultra-high-pressure wells where BHP exceeds the default
        (1011 bar)        

    Returns
    -------
    df : pd.DataFrame
        Indexed by 'depth (m)' (ascending, from 0 at the wellhead to tvd at
        BHP), with columns:

        density (kg/m3)  : fluid density from estimate_thp()
        viscosity (cP)   : fluid viscosity from estimate_thp()
        bhp-ph           : zero-rate (friction-free) pressure profile (bar);
                           BHP minus column weight above that depth. Its 1st value
                           at depth=0 is the static THP.
        bhp-ph+pf        : the same plus cumulative friction from that depth
                           down to tvd (bar). Its value at depth 0 is the
                           flowing THP. The value at 0 m is the flowing THP
    """
    df = estimate_thp(fluid, bhp, tvd, degC, all_points=True, p_max=p_max)
    df = df.set_index('depth (m)').sort_index(ascending=True)    
    dL = np.diff(df.index, append=np.nan)

    # average density/viscosity  between nodes
    rho = df['density (kg/m3)']
    mu = df['viscosity (cP)']
    rho = (rho.shift(-1) + rho) / 2  
    mu = (mu.shift(-1) + mu) / 2

    df['dpf'] = friction(mq, D, rho, mu, L=dL, roughness=roughness)['dp_bar']

    # setting the last dp_f value to 0
    df.iloc[-1,df.columns.get_loc('dpf')] = 0.0  
    df = df.rename(columns={'pressure (bar)':'bhp-ph'})
    df['bhp-ph+pf'] = df['bhp-ph'] + \
        df['dpf'].iloc[::-1].cumsum().iloc[::-1] # reversed cumsum
    # df = df.drop(columns=['dpf'])
    df = df[['density (kg/m3)', 'viscosity (cP)', 'bhp-ph', 'bhp-ph+pf']]
    return df

foo = estimate_thp_friction(tvd=2001.5)
foo

,density (kg/m3),viscosity (cP),bhp-ph,bhp-ph+pf
depth (m),,,,
0.000000,93.1862560503086,0.015607,38.534769,54.719288
0.999750,93.21563618256712,0.015607,38.543910,54.714507
1.999500,93.24502557791024,0.015607,38.553054,54.709734
2.999251,93.27442423925844,0.015608,38.562200,54.704968
3.999001,93.30383216953312,0.015608,38.571350,54.700210
...,...,...,...,...
1997.500999,855.8001164778124,0.082419,99.664183,99.670311
1998.500749,855.9088166466411,0.082442,99.748121,99.752717
1999.500500,856.0175306221128,0.082465,99.832070,99.835134


## without vs. with friction

In [17]:
# compact_legend=False: per-field legends (legend + legend2); True: single trace-type legend (legend3)
compact_legend = True
# compact_legend = False
ystr = 'depth (m)'
fields = ['TROLL', 'GULLFAKS', 'VOLVE', 'ULA', 'KVITEBJØRN']
field_colors = generate_rainbow_colors(len(fields))

style_specs = [(20, 1.2, 'solid'), (20, 1.2, 'dash')]
# style_specs = [(20, 1.2, 'solid'), (50, 1.2, 'dash'), (20, 1.0, 'dot')]
# fields = ['TROLL']
trace_count = len(style_specs)*len(style_specs[0])
field_count = len(fields)

fig = go.Figure()
x_max = 100
for i, (f,c) in enumerate(zip(fields, field_colors)):
    legend_id = 'legend3' if compact_legend else\
         ('legend' if i < 3 else 'legend2')
    p0,z,p_h = df.loc[f,['p0','depth','p_h']]
    for degC, p_mult, dsh in style_specs:

        bhp = p0*p_mult
        zdp=estimate_thp_friction('CO2', bhp=bhp, tvd=z, degC=degC, p_max=1201)
        zdp = zdp.reset_index()
        zdp[ystr] = zdp[ystr].astype(float)

        for xstr, dsh, leg in [('bhp-ph'   ,'solid', 'without friction'), 
                               ('bhp-ph+pf','dash',  'with friction')]:

            zdp[xstr] = zdp[xstr].astype(float)
            x_max = zdp[xstr].max() if zdp[xstr].max() > x_max else x_max
            name = f'{xstr}: T={degC}°C, BHP={p_mult}p0' \
                if p_mult > 1.0 else f'{xstr}: T={degC}°C, BHP=p0'

            fig.add_trace(go.Scatter(
                x=zdp[xstr], y=zdp[ystr], mode='lines', 
                line=dict(width=3, dash=dsh, color=c), 
                legend=legend_id, legendgroup=f, legendgrouptitle_text=f,
                name=leg, showlegend=(not compact_legend)
            ))
    # initial reservoir pressure point
    fig.add_trace(go.Scatter(
        x=[p0], y=[z], mode='markers', marker=dict(size=12, color=c),
        legend=legend_id, legendgroup=f, name='p0', showlegend=(not compact_legend)
    ))
    fig.add_annotation(
        x=1.1*p0, y=z+90, text=f, showarrow=False, arrowhead=1, font=dict(color=c))

if compact_legend:
    # dummy traces showing what each trace type looks like (p0 marker + line styles)
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers', marker=dict(size=12, color='white'),
        legend='legend3', name='initial pressure p0', 
        showlegend=True
    ))

    for xstr, dsh, name in [('bhp-ph'   ,'solid', 'without friction'), 
                            ('bhp-ph+pf','dash',  'with friction')]:
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines',
            line=dict(width=3, dash=dsh, color='white'),
            legend='legend3', name=name, showlegend=True
        ))

fig.update_yaxes(autorange='reversed', title=ystr)
fig.update_xaxes(title='pressure (bar)', 
                 range=[0, x_max*1.05]
                 )

legend_layout = (
    dict(legend3=dict(yanchor='top',y=0.95,xanchor='right',x=0.99,
                      font_size=28))
    if compact_legend else
    dict(legend =dict(yanchor='bottom',y=0.02,xanchor='left',x=0.02, 
                      font_size=20),
         legend2=dict(yanchor='top',y=0.99,xanchor='right',x=0.99,
                      font_size=20))
)
fig.update_layout(template='plotly_dark', **legend_layout, font_size = 20)
fig.write_html('temp.html',auto_open=True)

## sensitivityy to temperature

In [18]:
compact_legend = True
xstr = 'bhp-ph+pf'
ystr = 'depth (m)'
fields = ['TROLL', 'GULLFAKS', 'VOLVE','ULA','KVITEBJØRN']
field_colors = generate_rainbow_colors(len(fields))
style_specs = [(20, 1.2, 'solid'), (10, 1.2, 'dash'), (30, 1.2, 'dot')]
trace_count = len(style_specs)*len(style_specs[0])
field_count = len(fields)
fig = go.Figure()
x_max = 100
for i, (f,c) in enumerate(zip(fields, field_colors)):
    legend_id = 'legend3' if compact_legend else\
         ('legend' if i < 3 else 'legend2')
    p0,z,p_h = df.loc[f,['p0','depth','p_h']]
    for degC, p_mult, dsh in style_specs:

        bhp = p0*p_mult
        zdp=estimate_thp_friction(
            fluid='CO2', bhp=bhp, tvd=z, degC=degC, p_max=1201)

        zdp = zdp.reset_index()
        zdp[xstr] = zdp[xstr].astype(float)
        zdp[ystr] = zdp[ystr].astype(float)
        x_max = zdp[xstr].max() if zdp[xstr].max() > x_max else x_max

        name = f'T={degC}°C, BHP={p_mult}p0'\
              if p_mult > 1.0 else f'T={degC}°C, BHP=p0'

        fig.add_trace(go.Scatter(
            x=zdp[xstr], y=zdp[ystr], mode='lines', 
            line=dict(width=3, dash=dsh, color=c), 
            legend=legend_id, legendgroup=f, legendgrouptitle_text=f,
            name=name, showlegend=(not compact_legend)
        ))
    # initial reservoir pressure point
    fig.add_trace(go.Scatter(
        x=[p0], y=[z], mode='markers', marker=dict(size=12, color=c),
        legend=legend_id, legendgroup=f, name='p0', showlegend=(not compact_legend)
    ))
    fig.add_annotation(
        x=1.1*p0, y=z+90, text=f, showarrow=False, arrowhead=1, font=dict(color=c))

if compact_legend:
    # dummy traces showing what each trace type looks like (p0 marker + line styles)
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers', marker=dict(size=12, color='white'),
        legend='legend3', name='initial pressure p0', 
        showlegend=True
    ))
    for degC, p_mult, dsh in style_specs:
        name = f'T={degC}°C, BHP={p_mult}p0' if p_mult > 1.0 else f'T={degC}°C, BHP=p0'
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines',
            line=dict(width=3, dash=dsh, color='white'),
            legend='legend3', name=name, showlegend=True
        ))

fig.update_yaxes(autorange='reversed', title=ystr)
fig.update_xaxes(title='pressure (bar)', range=[0, x_max*1.05])

legend_layout = (
    dict(legend3=dict(yanchor='top',y=0.95,xanchor='right',x=0.99,
                      font_size=28))
    if compact_legend else
    dict(legend =dict(yanchor='bottom',y=0.02,xanchor='left',x=0.02, 
                      font_size=20),
         legend2=dict(yanchor='top',y=0.99,xanchor='right',x=0.99,
                      font_size=20))
)
fig.update_layout(template='plotly_dark', font_size=20, **legend_layout)
fig.write_html('temp.html',auto_open=True)

# simplest EE indicator for all reservoirs

In [19]:
df = pd.read_csv('./data/v2026.csv', index_col=0)
DEGC = [20]
for degC in DEGC:
    cn = f'WHP{degC}'
    cnf = f'WHP{degC}f'    
    df[cn] = np.nan
    for f in df.index:
        p0,z = df.loc[f,['p0','depth']]
        bhp = p0
        out = estimate_thp_friction(
            fluid='CO2',bhp=bhp,tvd=z,degC=degC)
        df.loc[f,cn] = out['bhp-ph'].iat[0]
        df.loc[f,cnf] = out['bhp-ph+pf'].iat[0]
    df[cn] = df[cn].round(1)
    df[cnf] = df[cnf].round(1)

In [20]:
df['EEI'], _ = estimate_compression_work(
    fluid='CO2', p=df['WHP20f'], degC=20, p_inc=1, p_max=500)
df['EEI'] = df['EEI']/1000
df['EEI'] = df['EEI'].round(0)

In [21]:
df.to_csv(f'./data/v2026+EEI.csv') # export

In [22]:
fig = px.scatter(
    df.reset_index(), x = 'WHP20', y = 'EEI', color='grad_p0', size='CO2 SC',
    hover_data=['p0','depth','field'], color_continuous_scale='rainbow', template='plotly_dark', size_max=100,
    labels={'WHP20': 'WHP (bar)', 'EEI':'compression work (MJ/tonne)'},
)
fig.update_layout(font_size = 20)
fig.update_traces(marker=dict(sizemin=3))
fig.show(renderer='browser')

In [23]:
fig = px.scatter(
    df.reset_index(), x = 'p0', y = 'EEI', color='grad_p0', size='CO2 SC',
    hover_data=['p0','depth','field'], color_continuous_scale='rainbow', template='plotly_white', # template='plotly_dark', 
    size_max=100,
    labels={'p0': 'initial pressure (bar)', 'EEI':'compression work (MJ/tonne)'}
)
fig.update_traces(marker=dict(sizemin=3))
fig.update_layout(font_size=20)
fig.show(renderer='browser')

In [24]:
def cross_plot(fig, xy_max, xy_min=0, percentages=[0, 5, 10, 15], log_axes=False):
    'adds crossplot lines to a Plotly figure'
    dashes = ['solid', 'dot', 'dash', 'longdash']

    if log_axes:
        xy = np.logspace(np.log10(xy_min), np.log10(xy_max))
    else:
        xy = np.linspace(xy_min, xy_max)

    for p, dash in zip(percentages, dashes):
        mp = 1 + p / 100
        mm = 1 - p / 100

        if p == 0:
            fig.add_trace(go.Scatter(
                x=xy, y=xy, mode='lines',
                name='1:1', line=dict(color='dimgray', width=1, dash=dash)
            ))
        else:
            fig.add_trace(go.Scatter(
                x=xy, y=xy * mp, mode='lines',
                name=f'+{p}%', showlegend=True,
                line=dict(color='dimgray', width=1, dash=dash)
            ))
            fig.add_trace(go.Scatter(
                x=xy, y=xy * mm, mode='lines',
                name=f'-{p}%', showlegend=False,
                line=dict(color='dimgray', width=1, dash=dash)
            ))

    if log_axes:
        fig.update_xaxes(type='log')
        fig.update_yaxes(type='log')

    fig.update_xaxes(range=[np.log10(xy_min), np.log10(xy_max)] if log_axes else [xy_min, xy_max])
    fig.update_yaxes(range=[np.log10(xy_min), np.log10(xy_max)] if log_axes else [xy_min, xy_max])

    fig.update_layout(
        legend=dict(yanchor='bottom', y=0.01, xanchor='right', x=0.99),
    )

    return fig

In [25]:
fig=px.scatter(df, x='WHP20', y='WHP20f')
fig = cross_plot(fig, xy_max=500, xy_min=40, log_axes=True)
fig.update_layout(
    xaxis_title='without friction (bar)', yaxis_title='with friction (bar)',
    font_size=20, template='plotly_white'
    )
fig.show(renderer='browser')
# fig.show()

In [26]:
df['WHP20'].min()

np.float64(51.5)

# leftovers

In [27]:
# pmax = 1201
# tt = np.arange(0,220,10) # temperature range (degC)
# pp = np.arange(1,pmax+1,10) # pressure range (bar)
# PVT_dict = {}
# f_fvf = {}
# for fluid, den_sc in zip(['CO2','CH4','H2'],[den_CO2_sc,den_CH4_sc,den_H2_sc]):
#     PVT_dict[fluid] = {}
#     DENS = np.empty(shape=(len(pp),len(tt)))
#     for n,T in enumerate(tt):
#         pvt_df=get_fluid_properties(
#             fluid=fluid,p_min=1, p_max=pmax, p_inc=10, degC=T, den_sc=den_sc,
#             process=False, # nist_folder='./data/NIST'
#             )   
#         PVT_dict[fluid][T] = pvt_df
#         pvt_df_=pvt_df.copy().drop_duplicates(subset=['Pressure (bar)'], keep='last')
#         # to avoid issues with duplicates around critical pressures
#         int_func = interp1d(pvt_df_['Pressure (bar)'], pvt_df_['FVF'],  kind='linear')
#         DENS[:,n] = int_func(pp)

#     f_fvf[fluid]=RegularGridInterpolator((pp,tt), DENS, method='pchip') 